In [1]:
import random
from pathlib import Path
from main import load_dotenv
from data_loader.raw_dataloader import RawDataloader
from models.ngram.knn import KNN
from models.ngram.model import Model
from collections import Counter, defaultdict

load_dotenv(Path("../.env"))

/home/skynet/research/MusicGeneration/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataloader = RawDataloader()
dataloader.prepare_dataset(representation="note_table")
loader = dataloader.loader(load_music=dataloader.load_note_table_music)

In [ ]:
tokens_by_pitch = defaultdict(Counter)

for batch in loader:
    for song in batch:
        note_tokens = zip(
            song["pitch"],
            song["velocity_bin"],
            song["delta_onset_bin"],
            song["duration_bin"],
        )

        for token in note_tokens:
            tokens_by_pitch[token[0]][token] += 1

knn = KNN(n=10, min_count=10)
neighbors = knn.get_nearest_neighbors(tokens_by_pitch)

In [ ]:
ngram_model = Model(loader=loader, neighbors=neighbors, n=4)
ngram_model.note_table_train()

In [ ]:
BOS = ("<BOS>", "<BOS>", "<BOS>", "<BOS>")
EOS = ("<EOF>", "<EOF>", "<EOF>", "<EOF>")

In [ ]:
from data_processing.music_representations.decoders import DecodeContext, create_decoder
from data_processing.music_representations.helpers.adapters import canonical_frames_to_midi


BASE_PATH = Path("/home/skynet/research/data/maestro/n_gram/note_table ngram=4 k =10/")
n_gen_songs = 250
i = 0

while i < n_gen_songs:
    tokens = (BOS, BOS, BOS)
    next_token = None
    while next_token != EOS:
        next_token = ngram_model.note_table_predict(tokens)[0]
        tokens += (next_token,)

    decoder = create_decoder("note_table", dataloader.config, dataloader.storage)
    frames = decoder.decode_tuples(tokens, DecodeContext(piece_id="ngram_sample"))

    for name, frame in frames.items():
        frame.write_parquet(BASE_PATH / f"generated_{i}.parquet")

    i+=1

In [ ]:
for line in decoder.tuple_contract().describe_loss():
    print(" -", line)